In [24]:
! kaggle competitions download -c playground-series-s6e3

playground-series-s6e3.zip: Skipping, found more recently modified local copy (use --force to force download)


In [36]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split

data_dir = "playground-series-s6e3"
target_col = "Churn"
id_col = "id"

train_df = pd.read_csv(f"{data_dir}/train.csv")
test_df = pd.read_csv(f"{data_dir}/test.csv")
sample_submission_df = pd.read_csv(f"{data_dir}/sample_submission.csv")

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("sample_submission shape:", sample_submission_df.shape)

with pd.option_context("display.max_columns", None):
    display(train_df.head())

train shape: (594194, 21)
test shape: (254655, 20)
sample_submission shape: (254655, 2)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [37]:
print("num_rows:", len(train_df))
print("num_columns:", train_df.shape[1])

for i, col in enumerate(train_df.columns):
    print(i, col, train_df[col].nunique())

num_rows: 594194
num_columns: 21
0 id 594194
1 gender 2
2 SeniorCitizen 2
3 Partner 2
4 Dependents 2
5 tenure 72
6 PhoneService 2
7 MultipleLines 3
8 InternetService 3
9 OnlineSecurity 3
10 OnlineBackup 3
11 DeviceProtection 3
12 TechSupport 3
13 StreamingTV 3
14 StreamingMovies 3
15 Contract 3
16 PaperlessBilling 2
17 PaymentMethod 4
18 MonthlyCharges 1921
19 TotalCharges 31910
20 Churn 2


In [40]:
# Split train into features and target. Test has no target column.
X_train_df = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].map({"No": 0, "Yes": 1}).astype("int64")
X_test_df = test_df.copy()

# One-hot encode only feature columns, and do it on train+test together
# so the encoded columns stay exactly the same.
combined_features = pd.concat([X_train_df, X_test_df], axis=0, ignore_index=True)
categorical_cols = combined_features.select_dtypes(include=["object"]).columns.tolist()
combined_features = pd.get_dummies(
    combined_features,
    columns=categorical_cols,
    dtype=int,
)

X_train_encoded = combined_features.iloc[:len(X_train_df)].copy()
X_test_encoded = combined_features.iloc[len(X_train_df):].copy()

# Drop id because it is usually not a useful model feature.
X_train_encoded = X_train_encoded.drop(columns=[id_col])
X_test_encoded = X_test_encoded.drop(columns=[id_col])

print("X_train_encoded shape:", X_train_encoded.shape)
print("X_test_encoded shape:", X_test_encoded.shape)
print("y_train shape:", y_train.shape)

#
with pd.option_context("display.max_columns", None):
    display(X_train_encoded.head())
    display(y_train.head())

X_train_encoded shape: (594194, 45)
X_test_encoded shape: (254655, 45)
y_train shape: (594194,)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,29,60.10,1653.85,0,1,0,1,0,1,0,1,1,0,0,1,0,0,0,0,1,1,0,0,0,0,1,0,0,1,1,0,0,1,0,0,0,1,0,0,1,0,0,0,1
1,0,58,69.50,3778.20,0,1,0,1,0,1,0,1,1,0,0,1,0,0,0,0,1,0,0,1,1,0,0,0,0,1,0,0,1,1,0,0,0,0,1,1,0,0,1,0,0
2,0,58,100.40,5841.35,0,1,0,1,1,0,0,1,0,0,1,0,1,0,1,0,0,0,0,1,1,0,0,1,0,0,0,0,1,0,0,1,1,0,0,0,1,0,0,1,0
3,0,1,69.70,70.70,1,0,1,0,1,0,0,1,1,0,0,0,1,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,1,0,0,1,0
4,0,1,70.45,70.45,1,0,1,0,1,0,0,1,1,0,0,0,1,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,1,0,0,1,0


0    0
1    0
2    0
3    1
4    1
Name: Churn, dtype: int64

In [ ]:
X_train_tensor = torch.tensor(X_train_encoded.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_encoded.values, dtype=torch.float32)

full_train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

train_size = int(len(full_train_dataset) * 0.8)
valid_size = len(full_train_dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, valid_dataset = random_split(
    full_train_dataset,
    [train_size, valid_size],
    generator=generator,
)

batch_size = 1024

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(X_test_tensor, batch_size=batch_size, shuffle=False)

print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)
print("X_test_tensor:", X_test_tensor.shape)
print("train size:", len(train_dataset))
print("valid size:", len(valid_dataset))
print("train batches:", len(train_dataloader))
print("valid batches:", len(valid_dataloader))

X_train_tensor: torch.Size([594194, 45])
y_train_tensor: torch.Size([594194, 1])
X_test_tensor: torch.Size([254655, 45])
train size: 475355
valid size: 118839
train batches: 465
valid batches: 117


In [33]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train_tensor.shape[1]


class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
        )
        self.activation = nn.ReLU()

    def forward(self, x):
        residual = x
        x = self.block(x)
        x = x + residual
        return self.activation(x)


class ChurnResidualFCN(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_blocks=3, dropout=0.2):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.res_blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)]
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        x = self.input_layer(x)
        x = self.res_blocks(x)
        x = self.head(x)
        return x


model = ChurnResidualFCN(input_dim=input_dim, hidden_dim=256, num_blocks=3, dropout=0.2).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)


print("device:", device)
print("input_dim:", input_dim)

device: cuda
input_dim: 45


In [ ]:
from tqdm.auto import tqdm

epochs = 50
best_valid_loss = float("inf")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_pbar = tqdm(train_dataloader, desc=f"Train Epoch {epoch + 1}/{epochs}", leave=True)
    for x, label in train_pbar:
        x = x.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = loss_fn(outputs, label)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)
        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).float()
        train_correct += (preds == label).sum().item()
        train_total += label.size(0)

        train_pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{train_correct / train_total:.4f}",
        })

    model.eval()
    valid_loss = 0.0
    valid_correct = 0
    valid_total = 0

    valid_pbar = tqdm(valid_dataloader, desc=f"Valid Epoch {epoch + 1}/{epochs}", leave=True)
    with torch.no_grad():
        for x, label in valid_pbar:
            x = x.to(device)
            label = label.to(device)

            outputs = model(x)
            loss = loss_fn(outputs, label)

            valid_loss += loss.item() * x.size(0)
            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()
            valid_correct += (preds == label).sum().item()
            valid_total += label.size(0)

            valid_pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{valid_correct / valid_total:.4f}",
            })

    avg_train_loss = train_loss / train_total
    avg_valid_loss = valid_loss / valid_total
    train_acc = train_correct / train_total
    valid_acc = valid_correct / valid_total

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"train_loss={avg_train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"valid_loss={avg_valid_loss:.4f}, valid_acc={valid_acc:.4f}"
    )

    if avg_valid_loss < best_valid_loss:
        best_valid_loss = avg_valid_loss
        torch.save(model.state_dict(), "best_churn_resfcn.pt")
        print("Saved best model.")

Train Epoch 1/50:   0%|          | 0/465 [00:00<?, ?it/s]

Valid Epoch 1/50:   0%|          | 0/117 [00:00<?, ?it/s]

Epoch 1/50 | train_loss=0.3103, train_acc=0.8557 | valid_loss=0.3315, valid_acc=0.8364
Saved best model.


Train Epoch 2/50:   0%|          | 0/465 [00:00<?, ?it/s]

Valid Epoch 2/50:   0%|          | 0/117 [00:00<?, ?it/s]

...

Train Epoch 7/50:   0%|          | 0/465 [00:00<?, ?it/s]

Valid Epoch 7/50:   0%|          | 0/117 [00:00<?, ?it/s]

Epoch 7/50 | train_loss=0.3086, train_acc=0.8568 | valid_loss=0.3406, valid_acc=0.8195


Train Epoch 8/50:   0%|          | 0/465 [00:00<?, ?it/s]

KeyboardInterrupt: 